In [1]:
from pathlib import Path
import numpy as np
import opengate as gate
import pandas as pd
from opengate.geometry.volumes import unite_volumes, subtract_volumes, intersect_volumes
import opengate_core as g4
from scipy.spatial.transform import Rotation
import tempfile
import k3d
import vtk
import os
import logging
import ipynbname

In [2]:
n_heads = 80

collimator_hole_size_mm = 2.3  # unit is mm
collimator_wall_thickness_mm = 2.0  # unit is mm
collimator_guide_length_mm = 3.0
detector_crystal_size_mm = 50.0

base_dir = Path(ipynbname.path()).parent.parents[2]
persistent_data_dir = base_dir / "persistent_data"
assert (
    persistent_data_dir.exists()
), f"Persistent data directory does not exist: {persistent_data_dir}"

xlsx_path = (
    persistent_data_dir
    / "spreadsheet"
    / "MDSL.excel80M10RFR.cut-plate.010.150roi.2.30pin.105ellipse.xlsx"
)
df_coords = pd.read_excel(
    xlsx_path, sheet_name="Coordinates"
)  # read the "Coordinates" sheet
df_coords.columns = df_coords.iloc[0]
df_coords = df_coords[1:]  # remove the first row which is now the header
df_coords = df_coords.reset_index(
    drop=True
)  # reset the index after removing the first row
df_coords = df_coords.apply(
    pd.to_numeric, errors="coerce"
)  # convert all columns to numeric, coercing errors to NaN
df_coords.columns.name = "Coordinates Sheet"

collimator_body_length_mm_np = df_coords["length of collimator"].values
collimator_hole_coords_mm = df_coords[
    [
        "x coordinate value at center of hole",
        "y coordinate value at center of hole",
        "z coordinate value at center of hole",
    ]
].values

if not isinstance(collimator_body_length_mm_np, np.ndarray):
    collimator_body_length_mm_np = np.asarray(collimator_body_length_mm_np)

if not isinstance(collimator_hole_coords_mm, np.ndarray):
    collimator_hole_coords_mm_np = np.asarray(collimator_hole_coords_mm)
else:
    collimator_hole_coords_mm_np = collimator_hole_coords_mm

hole_fov_center_distance_mm_np = np.linalg.norm(collimator_hole_coords_mm_np, axis=1)
azmuthal_angle_deg = (
    np.arctan2(collimator_hole_coords_mm_np[:, 1], collimator_hole_coords_mm_np[:, 0])
    * 180
    / np.pi
)
hole_fov_center_dist_xy_mm_np = np.linalg.norm(
    collimator_hole_coords_mm_np[:, :2], axis=1
)
polar_angle_deg = (
    np.arctan2(collimator_hole_coords_mm_np[:, 2], hole_fov_center_dist_xy_mm_np)
    * 180
    / np.pi
)
collimator_body_center_dist_mm_np = (
    hole_fov_center_distance_mm_np + collimator_body_length_mm_np
)
collimator_body_translation_mm = collimator_body_center_dist_mm_np.reshape(
    -1, 1
) * np.column_stack(
    (
        np.cos(np.radians(polar_angle_deg))
        * np.cos(np.radians(azmuthal_angle_deg)),
        np.cos(np.radians(polar_angle_deg))
        * np.sin(np.radians(azmuthal_angle_deg)),
        np.sin(np.radians(polar_angle_deg)),
    )
)



collimator_wall_thickness_mm_np = np.full((n_heads,), collimator_wall_thickness_mm)
collimator_body_inner_top_mm_np = np.full((n_heads,), detector_crystal_size_mm)
collimator_body_inner_bottom_mm_np = np.full((n_heads,), collimator_hole_size_mm)
collimator_body_outer_top_mm_np = (
    collimator_body_inner_top_mm_np + collimator_wall_thickness_mm_np * 2
)
collimator_body_outer_bottom_mm_np = (
    collimator_body_inner_bottom_mm_np + collimator_wall_thickness_mm_np * 2
)

collimator_guide_exit_angle_rad = np.arctan2(
    (collimator_body_inner_top_mm_np + collimator_body_inner_bottom_mm_np) * 0.5,
    collimator_body_length_mm_np,
)

# print("Collimator guide exit angle (degrees): ", collimator_guide_exit_angle_rad * 180 / np.pi)


collimator_guide_length_mm_np = np.full((n_heads,), collimator_guide_length_mm)

collimator_guide_distance_mm_np = hole_fov_center_distance_mm_np - collimator_guide_length_mm_np
collimator_guide_translation_mm = collimator_guide_distance_mm_np.reshape(
    -1, 1
) * np.column_stack(
    (
        np.cos(np.radians(polar_angle_deg))
        * np.cos(np.radians(azmuthal_angle_deg)),
        np.cos(np.radians(polar_angle_deg))
        * np.sin(np.radians(azmuthal_angle_deg)),
        np.sin(np.radians(polar_angle_deg)),
    )
)

collimator_guide_inner_top_mm_np = np.full((n_heads,), collimator_hole_size_mm)
collimator_guide_outer_top_mm_np = (
    collimator_guide_inner_top_mm_np + collimator_wall_thickness_mm_np * 2
)
collimator_guide_inner_bottom_mm_np = (
    collimator_guide_inner_top_mm_np
    + np.tan(collimator_guide_exit_angle_rad) * collimator_guide_length_mm_np * 2
)
collimator_guide_outer_bottom_mm_np = (
    collimator_guide_inner_bottom_mm_np + collimator_wall_thickness_mm_np * 2
)

In [ ]:
sim = gate.Simulation()

for i in range(n_heads):
    collimator_body_inner = gate.geometry.volumes.TrdVolume(  # type: ignore
        name=f"CollimatorBody_{i+1}"
    )
    collimator_body_outer = gate.geometry.volumes.TrdVolume(  # type: ignore
        name=f"CollimatorBody_outer_{i+1}"
    )
    collimator_body_outer.dx1 = collimator_body_outer_top_mm_np[i] * 0.5
    collimator_body_outer.dy1 = collimator_body_outer_top_mm_np[i] * 0.5
    collimator_body_outer.dx2 = collimator_body_outer_bottom_mm_np[i] * 0.5
    collimator_body_outer.dy2 = collimator_body_outer_bottom_mm_np[i] * 0.5
    collimator_body_outer.dz = collimator_body_length_mm_np[i]
    collimator_body_inner.dx1 = collimator_body_inner_top_mm_np[i] * 0.5
    collimator_body_inner.dy1 = collimator_body_inner_top_mm_np[i] * 0.5
    collimator_body_inner.dx2 = collimator_body_inner_bottom_mm_np[i] * 0.5
    collimator_body_inner.dy2 = collimator_body_inner_bottom_mm_np[i] * 0.5
    collimator_body_inner.dz = collimator_body_length_mm_np[i]+0.2
    collimator_body = subtract_volumes(
        collimator_body_outer, collimator_body_inner, new_name=f"CollimatorBody_{i+1}"
    )
    collimator_body.mother = "world"
    # Collimator Guide
    collimator_guide_inner = gate.geometry.volumes.TrdVolume(  # type: ignore
        name=f"CollimatorGuide_{i+1}"
    )
    collimator_guide_outer = gate.geometry.volumes.TrdVolume(  # type: ignore
        name=f"CollimatorGuide_outer_{i+1}"
    )
    collimator_guide_outer.dx1 = collimator_guide_outer_top_mm_np[i] * 0.5
    collimator_guide_outer.dy1 = collimator_guide_outer_top_mm_np[i] * 0.5
    collimator_guide_outer.dx2 = collimator_guide_outer_bottom_mm_np[i] * 0.5
    collimator_guide_outer.dy2 = collimator_guide_outer_bottom_mm_np[i] * 0.5
    collimator_guide_outer.dz = collimator_guide_length_mm_np[i]
    collimator_guide_inner.dx1 = collimator_guide_inner_top_mm_np[i] * 0.5
    collimator_guide_inner.dy1 = collimator_guide_inner_top_mm_np[i] * 0.5
    collimator_guide_inner.dx2 = collimator_guide_inner_bottom_mm_np[i] * 0.5
    collimator_guide_inner.dy2 = collimator_guide_inner_bottom_mm_np[i] * 0.5
    collimator_guide_inner.dz = collimator_guide_length_mm_np[i]+0.2
    collimator_guide = subtract_volumes(
        collimator_guide_outer,
        collimator_guide_inner,
        new_name=f"CollimatorGuide_{i+1}",
    )

    # sim.add_volume(collimator_body)
    # sim.add_volume(collimator_guide)
    # collimator_guide.mother = f"CollimatorBody_{i+1}"
    # collimator_guide.translation = [0, 0, collimator_body_length_mm_np[i] + collimator_guide_length_mm_np[i]]
    collimator = unite_volumes(
        collimator_body,
        collimator_guide,
        new_name=f"Collimator_{i+1}",
        translation=[
            0,
            0,
            collimator_body_length_mm_np[i] + collimator_guide_length_mm_np[i],
        ],
    )
    sim.add_volume(collimator, name=f"Collimator_{i+1}")
    collimator.mother = "world"
    collimator.translation = collimator_body_translation_mm[i]
    # First rotate around x axis by 90 degrees
    rx_0 = Rotation.from_euler("x", -90, degrees=True).as_matrix()
    rz_0 = Rotation.from_euler("z", 90, degrees=True).as_matrix()
    # Then rotate around z axis by the azmuthal angle
    rz_1 = Rotation.from_euler("z", azmuthal_angle_deg[i], degrees=True).as_matrix()

    # # Then rotate around y axis by the polar angle
    rx_1 = Rotation.from_euler("x", -polar_angle_deg[i], degrees=True).as_matrix()
    r = rz_1@rz_0@rx_1@rx_0
    collimator.rotation = r

front_shielding = gate.geometry.volumes.TesselatedVolume(name="FrontShielding")  # type: ignore
front_shielding.mother = "world"
front_shielding.file_name = str(
    (persistent_data_dir / "stl" / "front_shielding.stl").as_posix()
)
front_shielding.origin_at_cog = False
print("front shielding volume: ", front_shielding.solid_info.cubic_volume)
front_shielding.rotation = Rotation.from_euler("z", 90, degrees=True).as_matrix()
sim.add_volume(front_shielding)

sim.volume_manager.dump_volume_tree()

temp_vrml = tempfile.NamedTemporaryFile(suffix=".wrl", dir="/dev/shm", delete=False)
temp_vrml.close()  # Close it so Geant4 can write to it
sim.user_info.visu = True
sim.user_info.visu_type = "vrml_file_only"
sim.user_info.visu_filename = temp_vrml.name
sim.run()


# Suppress INFO logs from K3D's helper module
logging.getLogger("k3d.helpers").setLevel(logging.WARNING)
importer = vtk.vtkVRMLImporter()
importer.SetFileName(temp_vrml.name)
importer.Update()

# 2. Access the renderer to retrieve the geometric shapes (actors)
render_window = importer.GetRenderWindow()
renderer = render_window.GetRenderers().GetFirstRenderer()
actors = renderer.GetActors()

# 3. Initialize an empty K3D plot
plot = k3d.plot(camera_auto_fit=False)

# [pos_x, pos_y, pos_z, target_x, target_y, target_z, up_x, up_y, up_z]
plot.camera = [
    300,
    300,
    300,  # Camera Position
    0,
    0,
    0,  # Looking at the origin (center of your geometry)
    0,
    0,
    1,  # Z-axis is "up"
]

# 4. Iterate over each piece of the geometry, extract it, and add to K3D
actors.InitTraversal()
number_of_actors = actors.GetNumberOfItems()
print(f"Total number of actors in the scene: {number_of_actors}")

all_actors = [actors.GetNextActor() for i in range(number_of_actors)]

for i, actor in enumerate(all_actors):
    # if i<62 and (i == 0 or i == number_of_actors-1 or (i-1) % 3 !=2):
    #     continue  # Skip the first and last actors
    # Skip the first
    if i == 0:
        continue

    polydata = actor.GetMapper().GetInput()
    # --- Edge Extraction and Tubing ---

    # 1. Clean the mesh (weld duplicate vertices)
    cleaner = vtk.vtkCleanPolyData()
    cleaner.SetInputData(polydata)
    cleaner.Update()

    # 2. Extract the feature edges from the cleaned mesh
    edge_filter = vtk.vtkFeatureEdges()
    edge_filter.SetInputConnection(cleaner.GetOutputPort())
    edge_filter.BoundaryEdgesOn()
    edge_filter.FeatureEdgesOn()
    edge_filter.SetFeatureAngle(60.0)
    edge_filter.NonManifoldEdgesOff()
    edge_filter.ManifoldEdgesOff()
    edge_filter.Update()

    # 3. Turn the 1D lines into 3D tubes for K3D rendering
    tube_filter = vtk.vtkTubeFilter()
    tube_filter.SetInputConnection(edge_filter.GetOutputPort())
    tube_filter.SetRadius(
        0.2
    )  # *** ADJUST THIS RADIUS based on your geometry scale ***
    tube_filter.SetNumberOfSides(6)
    tube_filter.Update()

    # Get the thickened sharp edges
    sharp_edges_polydata = tube_filter.GetOutput()
    if sharp_edges_polydata.GetNumberOfPoints() == 0:
        print(f"Actor {i} has no sharp edges, skipping edge rendering.")
    else:
        # Add the sharp edges to the plot (in red)
        plot += k3d.vtk_poly_data(sharp_edges_polydata, color=0xFF0000)

    # ----------------------------------
    if i == number_of_actors - 1:
        # For the last actor, render it in a different color (e.g., green)
        plot += k3d.vtk_poly_data(polydata, color=0x00FF00, wireframe=False, opacity=1)
    else:
    # Plot the original surfaces in transparent blue
    plot += k3d.vtk_poly_data(polydata, color=0x0072BD, wireframe=False, opacity=1)

# 5. Render the final output
plot.display()
os.remove(temp_vrml.name)

front shielding volume:  2657461.815675939
⚠️ No configured source, no particle will be generated.
Simulation: create RunManager (single thread)
Simulation: initialize Geometry
Simulation: initialize Physics
Simulation: initialize Sources
Simulation: initialize Visualization
Simulation: initialize G4RunManager
Simulation: initialize PhysicsEngine
Simulation: initialize Actors
Simulation: check volumes overlap
--------------------------------------------------------------------------------
Simulation: START 
Simulation: STOP. Run: 1. Time: 0.5 seconds.
--------------------------------------------------------------------------------
⚠️ One warning occurred in this simulation:
⚠️ (1) No configured source, no particle will be generated.
Total number of actors in the scene: 83


2026-04-08 12:36:24.650 (   2.004s) [    7E127D04B740]vtkXOpenGLRenderWindow.:1460  WARN| bad X server connection. DISPLAY=


Actor 82 has no sharp edges, skipping edge rendering.


Output()